In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from gen_ai_hub.proxy.langchain.openai import ChatOpenAI
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool
import requests

In [ ]:
# Define tools
@tool
def add(a: float, b: float) -> float:
    """Add two numbers together."""
    return a + b

@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers together."""
    return a * b

@tool
def get_weather(city: str) -> str:
    """Get the current weather for a given city."""
    url = f"https://wttr.in/{city}?format=3"
    response = requests.get(url)
    if response.status_code == 200:
        return response.text
    return f"Could not retrieve weather for {city}."

tools = [add, multiply, get_weather]

In [ ]:
# LLM with tools bound
llm = ChatOpenAI(proxy_model_name='gpt-4o')
llm_with_tools = llm.bind_tools(tools)

In [ ]:
# Define state
class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [ ]:
# Define nodes
def llm_node(state: AgentState):
    response = llm_with_tools.invoke(state['messages'])
    return {'messages': [response]}

tool_node = ToolNode(tools)

In [ ]:
# Build graph
graph = StateGraph(AgentState)

graph.add_node('llm_node', llm_node)
graph.add_node('tools', tool_node)

graph.add_edge(START, 'llm_node')
graph.add_conditional_edges('llm_node', tools_condition)
graph.add_edge('tools', 'llm_node')

agent = graph.compile()
agent

In [ ]:
# Test: math
result = agent.invoke({'messages': [HumanMessage(content='What is 25 multiplied by 4, then add 18?')]})
print(result['messages'][-1].content)

In [ ]:
# Test: weather
result = agent.invoke({'messages': [HumanMessage(content='What is the weather in Berlin?')]})
print(result['messages'][-1].content)